In [26]:
import pandas as pd
import numpy as np

In [27]:
order_level = pd.read_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\data\processed\order_level_clean.csv")
item_level = pd.read_csv(r"D:\projects\Brazilian E-Commerce Public Dataset\data\processed\order_level_clean.csv")

In [28]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for c in date_cols:
    if c in order_level.columns:
        order_level[c] = pd.to_datetime(order_level[c], dayfirst=True, errors="coerce")
                                                        

print("Dates converted with dayfirst=True")
print(order_level[date_cols].dtypes)

Dates converted with dayfirst=True
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


Order-level Engineered Features

In [30]:

# Order-level engineered features


# 1. Delivery time (days) — purchase to delivery
order_level["delivery_days"] = (
    order_level["order_delivered_customer_date"] - order_level["order_purchase_timestamp"]
).dt.days

# 2. Delivery delay vs estimate (negative = early, positive = late)
order_level["delivery_vs_estimate_days"] = (
    order_level["order_delivered_customer_date"] - order_level["order_estimated_delivery_date"]
).dt.days

# 3. Is late delivery flag
order_level["is_late"] = (order_level["delivery_vs_estimate_days"] > 0).astype(int)

# 4. Payment behavior: uses installments?
order_level["uses_installments"] = (order_level["max_installments"] > 1).astype(int)

# 5. Order value bucket (low/medium/high)
order_level["order_value_segment"] = pd.cut(
    order_level["total_payment_value"],
    bins=[0, 50, 150, 500, np.inf],
    labels=["low", "medium", "high", "premium"]
)

# 6. Purchase time parts (for trend features)
order_level["purchase_year"] = order_level["order_purchase_timestamp"].dt.year
order_level["purchase_month"] = order_level["order_purchase_timestamp"].dt.month
order_level["purchase_dayofweek"] = order_level["order_purchase_timestamp"].dt.dayofweek

print("Order-level features added")
print(order_level[["delivery_days","delivery_vs_estimate_days","is_late","uses_installments","order_value_segment"]].head())

Order-level features added
   delivery_days  delivery_vs_estimate_days  is_late  uses_installments  \
0            6.0                      -28.0        0                  1   
1            9.0                      -16.0        0                  1   
2           10.0                      -19.0        0                  1   
3           25.0                        3.0        1                  0   
4           11.0                      -21.0        0                  1   

  order_value_segment  
0              medium  
1                high  
2              medium  
3                high  
4              medium  
